# Tear down the cluster (PDF Step 5)

## Objective
Delete everything `01`/`02` created — the cluster, then the Filestore instance,
then the code-transfer bucket. Accelerators bill for as long as the cluster
exists, so don't skip this.

In [1]:
import yaml, pathlib
cfg = yaml.safe_load(open("../configs/cluster_config.yaml"))
PROJECT = cfg["project"]["id"]; ZONE = cfg["project"]["zone"]
FILESTORE_ID = cfg["filestore"]["instance_id"]
BUCKET = cfg["bucket"]["name"] or f"{PROJECT}-vtc-temp"
print("project:", PROJECT, "| filestore:", FILESTORE_ID, "| bucket:", BUCKET)

project: sandbox-401718 | filestore: filestore-test | bucket: sandbox-401718-vtc-temp


## 1. Delete the cluster

`delete_cluster.sh` reads the cluster id back from `.vtc_cluster_state` (or pass
`--cluster-id`). It removes only the cluster — Filestore/bucket are handled below.

In [2]:
!bash ../scripts/delete_cluster.sh --config ../configs/cluster_config.yaml --state-in .vtc_cluster_state

==> deleting cluster 'vtcgemma' in sandbox-401718/us-east4
{
  "name": "projects/757654702990/locations/us-east4/operations/5851522680259346432",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.aiplatform.v1beta1.DeleteModelDevelopmentClusterOperationMetadata",
    "genericMetadata": {
      "createTime": "2026-06-26T18:26:34.658211Z",
      "updateTime": "2026-06-26T18:26:34.658211Z"
    }
  }
}

==> delete requested. Track with: gcloud ai operations list --region=us-east4


## 2. Delete Filestore + bucket (PDF Steps 5.2–5.3)

> These hold your `/home` and staged code. Make sure you've copied off anything
> you want to keep — this is irreversible.

In [3]:
!gcloud filestore instances delete {FILESTORE_ID} --zone={ZONE} --project={PROJECT} --quiet
!gcloud storage rm -r gs://{BUCKET}
print("Teardown complete.")

Waiting for [operation-1782498399728-6552c3fedb262-51501e80-39b02664] to finish
...done.                                                                       
Removing objects:
  Completed 0                                                                  
Removing buckets:
Removing gs://sandbox-401718-vtc-temp/...                                      
  Completed 1/1                                                                
Teardown complete.


> **Using the Terraform path instead?** Don't mix them — run
> `terraform destroy` from `terraform/`, which removes the cluster (via the
> destroy provisioner), Filestore, and bucket together.